# Discovering the PDE parameters using a PINN

In this notebook we are going to implement the data-driven discovery of partial differential equations with the continuous time model of the Physics-Informed Neural Networks, which it was presented in Raissi, Perdikaris & Karniadakis (2019), *Journal of Computational Physics 378*, 686–707.

In the previous notebook (*Solving the Burgers Equation using a PINN*) the equation was known and we only wanted to find $u(t, x)$. Now the problem is the opposite: we only have noisy observations of $u(t, x)$ and the coefficients of the equation are unknown. So the idea is to represent $u(t, x)$ using a neural network $u_{\theta}(t, x)$ and to treat the coefficients $\lambda_1$ and $\lambda_2$ as extra trainable parameters of the model. Through automatic differentiation we derive the network with regard to its inputs $(t, x)$ and we calculate the PDE residual $f(t, x)$, which now depends on $\lambda_1$ and $\lambda_2$.

To train the network we are going to minimize the loss against the observations and the PDE residual, both evaluated in the same observation points:
$$
    \mathcal{L}_{MSE} = \mathcal{L}_{MSE_{u}} + \mathcal{L}_{MSE_{f}},
$$
where
$$
    \mathcal{L}_{MSE_{u}} = \frac{1}{N}\sum_{i=1}^{N}\left|u(t^i,x^i)-u_{obs}^i\right|^2, \qquad \mathcal{L}_{MSE_{f}} = \frac{1}{N}\sum_{i=1}^{N}\left|f(t^i,x^i)\right|^2.
$$
Notice that there are no initial or boundary conditions here and no separate collocation points. Because $\lambda_1$ and $\lambda_2$ are optimized together with the weights of the network, when the training ends they are our estimation of the coefficients of the equation.

## 1. Initial configuration

Here we are only importing the basic libraries and the modules of the project which are in `src/`.

In [ ]:
# To import my files
import sys
import os

src = os.path.abspath("./src")

if src not in sys.path:
    sys.path.append(src)

In [ ]:
# Imports
import torch
from tqdm import tqdm
from matplotlib import pyplot as plt

from src.model import PINNDiscovery
from src.physics import compute_pde_residual_discovery
from src.dataset import generate_noisy_data

## 2. Experiment parameters

In this code we are considering the essential parameters of the project: `X_MIN`, `X_MAX` and `T_MIN`, `T_MAX` are just the spatial-temporal domain, those are $[-10, 10]$ and $[0, 1]$ respectively; `N_OBS` is the number of noisy observation points; `LAYERS` is a list with the neurons number for each layer in the neural network; we are using an Adam with `LEARNING_RATE` of $10^{-3}$ and trained during 20000 `EPOCHS`; lastly, we are training the model in `DEVICE`.

In [ ]:
# Parameters of the exp.

# Domain
X_MIN, X_MAX = -10.0, 10.0
T_MIN, T_MAX = 0.0, 1.0

# Number of observation points
N_OBS = 500

# Neural network architecture
LAYERS = [2, 20, 20, 20, 20, 20, 20, 20, 20, 1]

# Optimization hyperparameters
LEARNING_RATE = 1e-3
EPOCHS = 20000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

This code is only to fix a seed.

In [ ]:
# Fix seed
torch.manual_seed(151)

## 3. Train data

Here we have the `generate_noisy_data` which generates the observations: `(x_data, t_data)` are the points where we measure and `u_obs` are the values of $u$ in those points, with noise added. This is the only information about the solution that the model is going to see.

In [ ]:
# Generate data
(x_data, t_data), u_obs = generate_noisy_data(N_OBS, X_MIN, X_MAX, T_MIN, T_MAX, DEVICE)

## 4. Model, Optimizer Algorithm and Train loop

In this code we instance the model `PINNDiscovery` which has the neural network and the two trainable coefficients `lambda_1` and `lambda_2`. All of them are trained with the same Adam optimizer. We also create three lists to save the loss and the value of each coefficient at every epoch.

In [ ]:
# Model
model = PINNDiscovery(LAYERS).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

loss_history = []
lambdas_1_history = []
lambdas_2_history = []

This is basically the train loop. In each epoch `compute_pde_residual_discovery` returns the prediction `u_pred` and the residual `f_pred`, then we calculate $\mathcal{L}_{MSE_u}$ against the observations and $\mathcal{L}_{MSE_f}$ against zero. Here it is only interesting to know that this code is heavier than the one in the solutions notebook (20000 epochs), it took roughly 7 minutes in my machine.

In [ ]:
# Train loop
model.train()

for _ in tqdm(range(1, EPOCHS + 1), desc="Train progress", unit=" Epoch"):
    optimizer.zero_grad()
    
    u_pred, f_pred = compute_pde_residual_discovery(model, t_data, x_data)
    
    loss_u = torch.mean((u_pred - u_obs)**2)
    loss_f = torch.mean(f_pred**2)
    
    loss = loss_u + loss_f
    
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    lambdas_1_history.append(model.lambda_1.item())
    lambdas_2_history.append(model.lambda_2.item())

## Results

First we can show how the coefficients $\lambda_1$ and $\lambda_2$ change during the training. The values at the end are the ones that the model discovered for the equation.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(lambdas_1_history, color="#c20202", label=r"$\lambda_1$")
plt.plot(lambdas_2_history, color="#0f02c2", label=r"$\lambda_2$")


plt.grid(True, which="both", ls="--")

plt.xlabel('Iterations')

plt.legend()
plt.tight_layout()

plt.show()

Here we can show the total loss $\mathcal{L}_{MSE_u} + \mathcal{L}_{MSE_f}$ per epoch, in logarithm scale.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(loss_history, label='Total Loss')
plt.yscale('log')

plt.xlabel('Iterations')
plt.ylabel('MSE (Log Scale)')

plt.grid(True, which="both", ls="--")

plt.legend()
plt.tight_layout()

plt.show()